# TRI-X-CDSS Quickstart

This starter notebook turns the framework modules into an interactive walkthrough covering triage, SRGL screening, DRAS-5 classification, and ORASR routing.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime

sys.path.insert(0, str(Path.cwd().parent / 'src'))

In [ ]:
from trix_cdss.core.triage import (
    DizzinessSymptoms,
    PatientRiskFactors,
    VitalSigns,
    map_esi_to_dras,
    perform_dizziness_triage,
)
from trix_cdss.core.srgl import perform_red_flag_screening
from trix_cdss.core.dras5 import DRAS5Features, classify_urgency_level
from trix_cdss.core.orasr import PatientContext, ResourceAvailability, route_to_care_pathway
from trix_cdss.core.titrate import ClinicalFindings, TimePoint, perform_time_bounded_assessment

## Build a representative dizziness case

In [ ]:
vitals = VitalSigns(heart_rate=92, systolic_bp=168, diastolic_bp=96, respiratory_rate=18, temperature=36.8, oxygen_saturation=97)
symptoms = DizzinessSymptoms(
    symptom_description='continuous vertigo',
    duration_hours=1.5,
    episode_duration_seconds=None,
    diplopia=True,
    ataxia=True,
    nausea_vomiting=True,
    continuous_vertigo=True,
)
risk_factors = PatientRiskFactors(age=72, atrial_fibrillation=True, hypertension=True, prior_stroke_tia=False)

## Triage

In [ ]:
triage = perform_dizziness_triage(vitals, symptoms, risk_factors)
{
    'esi_level': triage.esi_level.value,
    'rationale': triage.rationale,
    'resources_needed': triage.estimated_resources,
}

## SRGL red-flag screening

In [ ]:
screening = perform_red_flag_screening(
    symptoms={'headache': False, 'diplopia': True, 'ataxia': True},
    vital_signs={'heart_rate': 92, 'systolic_bp': 168},
    neurological_exam={'diplopia': True, 'ataxia': True, 'dysarthria': False},
    medical_history={'anticoagulation': False},
)
{
    'critical_red_flags': screening.has_critical_red_flags(),
    'safe_to_proceed': screening.is_safe_to_proceed(),
    'recommendations': screening.recommendations,
}

## TiTrATE assessment

In [ ]:
findings_t0 = ClinicalFindings(
    time_point=TimePoint.T0,
    timestamp=datetime.now(),
    heart_rate=92,
    blood_pressure_sys=168,
    blood_pressure_dia=96,
    hints_performed=True,
    hints_hit_result='negative',
    hints_nystagmus='direction_changing',
    hints_skew=True,
    ataxia=True,
    symptom_severity=8,
)
titrate = perform_time_bounded_assessment(
    patient_id='P001',
    initial_presentation_time=datetime.now(),
    time_point=TimePoint.T0,
    findings=findings_t0,
)
titrate.get_assessment_summary()

## DRAS-5 classification

In [ ]:
initial_dras = map_esi_to_dras(triage.esi_level, risk_factors.calculate_stroke_risk_score())
features = DRAS5Features(
    age=72,
    gender='male',
    atrial_fibrillation=True,
    hypertension=True,
    symptom_duration_hours=1.5,
    symptom_severity=8,
    hints_performed=True,
    hints_central=True,
    ataxia=True,
    diplopia=True,
    systolic_bp=168,
    heart_rate=92,
    stroke_probability=0.86,
    continuous_vertigo=True,
)
classification = classify_urgency_level(
    features,
    esi_level=triage.esi_level.value,
    red_flags_critical=screening.has_critical_red_flags(),
    titrate_risk_score=8.5,
)
{
    'initial_dras': initial_dras,
    'final_level': classification.level.value,
    'care_setting': classification.get_target_care_setting(),
    'rationale': classification.rationale,
}

## ORASR routing

In [ ]:
patient_context = PatientContext(can_travel_independently=False, has_transportation=True, home_support=True)
resources = ResourceAvailability(imaging_ct_available=True, neurology_available=True, stroke_team_available=True)
routing = route_to_care_pathway(
    dras_level=classification.level.value,
    patient_context=patient_context,
    resource_availability=resources,
    clinical_features={'stroke_probability': 0.86, 'hints_central': True, 'severe': True},
    red_flags_present=screening.has_critical_red_flags(),
)
routing.get_routing_summary()